In [100]:
from sympy import *
from IPython.display import display

init_printing(use_latex="mathjax")

In [101]:
phi, theta, delta = symbols('phi theta delta', real=True)

phi_EOM1 = symbols(r'\phi_{1}', real=True)
phi_EOM2 = symbols(r'\phi_{2}', real=True)

Ex, Ey = symbols('E_x E_y', complex=True)

ax, ay = symbols('a_x a_y', real = True)

# Some math definitions:

t = 1/sqrt(2)
r = I/sqrt(2)

In [102]:

Ein = Matrix([
    [ax * exp(I * delta)],
    [ay]
])

display(Ein)

⎡    ⅈ⋅δ⎤
⎢aₓ⋅ℯ   ⎥
⎢       ⎥
⎣  a_y  ⎦

In [103]:
def Mirror():

    return eye(2)
    

def PBS_T():

    return Matrix([
        [1,0],
        [0,0]
    ])


def PBS_R():

    return Matrix([
        [0,0],
        [0,1]
    ])
    

def BS():

    return Matrix([
        [1,I],
        [I,1]
    ]) / sqrt(2)


def BS(EA, EB):

    t = 1/sqrt(2)
    r = I/sqrt(2)

    EC = simplify(t*EA + r*EB)
    ED = simplify(r*EA + t*EB)

    return EC, ED


def EOM1(phi_EOM1):

    return Matrix([
        [exp(I*phi_EOM1),0],
        [0,1]
    ])

def EOM2(phi_EOM2):

    return Matrix([
        [exp(I*phi_EOM2),0],
        [0,1]
    ])

In [104]:
def propagate(E, *elements):

    for M in elements:
        E = simplify(M*E)

    return simplify(E)

In [105]:
armA = propagate(
    Ein,
    PBS_T(),
    Mirror(),
    EOM1(phi_EOM1),
)

display(armA)

⎡    ⅈ⋅(\phi_{1} + δ)⎤
⎢aₓ⋅ℯ                ⎥
⎢                    ⎥
⎣         0          ⎦

In [106]:
armB = propagate(
    Ein,
    PBS_R(),
    Mirror(),
)

display(armB)

⎡ 0 ⎤
⎢   ⎥
⎣a_y⎦

In [107]:
armC = simplify(r*armA + t*armB)

armC = propagate(
    armC,
    Mirror(),
    EOM2(phi_EOM2)
)

display(armC)

⎡         ⅈ⋅(\phi_{1} + \phi_{2} + δ)⎤
⎢√2⋅ⅈ⋅aₓ⋅ℯ                           ⎥
⎢────────────────────────────────────⎥
⎢                 2                  ⎥
⎢                                    ⎥
⎢               √2⋅a_y               ⎥
⎢               ──────               ⎥
⎣                 2                  ⎦

In [108]:
armD = simplify(r*armB + t*armA)

armD = propagate(
    armD,
    Mirror(),
)

display(armD)

⎡       ⅈ⋅(\phi_{1} + δ)⎤
⎢√2⋅aₓ⋅ℯ                ⎥
⎢───────────────────────⎥
⎢           2           ⎥
⎢                       ⎥
⎢       √2⋅ⅈ⋅a_y        ⎥
⎢       ────────        ⎥
⎣          2            ⎦

In [112]:
armE = simplify(r*armC + t*armD)

display(armE)
print(armE)

⎡   ⎛ ⅈ⋅(\phi_{1} + δ)    ⅈ⋅(\phi_{1} + \phi_{2} + δ)⎞⎤
⎢aₓ⋅⎝ℯ                 - ℯ                           ⎠⎥
⎢─────────────────────────────────────────────────────⎥
⎢                          2                          ⎥
⎢                                                     ⎥
⎣                        ⅈ⋅a_y                        ⎦

Matrix([[a_x*(exp(I*(\phi_{1} + delta)) - exp(I*(\phi_{1} + \phi_{2} + delta)))/2], [I*a_y]])


In [110]:
armF = simplify(r*armD + t*armC)

display(armF)

⎡     ⎛ ⅈ⋅(\phi_{1} + δ)    ⅈ⋅(\phi_{1} + \phi_{2} + δ)⎞⎤
⎢ⅈ⋅aₓ⋅⎝ℯ                 + ℯ                           ⎠⎥
⎢───────────────────────────────────────────────────────⎥
⎢                           2                           ⎥
⎢                                                       ⎥
⎣                           0                           ⎦

In [98]:
from sympy import (
    symbols, Matrix, exp, I,
    simplify, powsimp, factor, expand,
    Abs, arg
)

###############################################################################
# 1. Phase parameters
###############################################################################

phi1, phi2, delta = symbols(r'\phi_1 \phi_2 \delta', real=True)

###############################################################################
# 2. CLEAN PHASES (UPGRADED)
###############################################################################

def clean_phases(expr):
    """
    Strong canonicalization of optical phase expressions:
    - merges exponentials
    - expands products
    - removes redundant structure
    """

    expr = expr.rewrite(exp)

    # key upgrade: expansion BEFORE powsimp (important for nested interferometers)
    expr = expand(expr)

    # merges exp(a)*exp(b) -> exp(a+b)
    expr = powsimp(expr, force=True)

    expr = factor(expr)
    expr = simplify(expr)

    return expr

###############################################################################
# 3. GLOBAL PHASE REMOVAL (IMPORTANT FOR INTERFEROMETRY VISUALIZATION)
###############################################################################

def remove_global_phase(E):
    """
    Normalizes Jones vector so global phase is removed.
    This is what makes polarization visually interpretable.
    """

    E0 = E[0]
    phase0 = arg(E0)

    return simplify(E * exp(-I * phase0))

###############################################################################
# 4. PHASOR DECOMPOSITION (ROBUST)
###############################################################################

def phasor_decompose(E):
    """
    Returns:
        amplitude vector
        phase vector (relative, stable)
    """

    Ex, Ey = E[0], E[1]

    amplitudes = Matrix([
        simplify(Abs(Ex)),
        simplify(Abs(Ey))
    ])

    phases = Matrix([
        simplify(arg(Ex)),
        simplify(arg(Ey))
    ])

    return amplitudes, phases

###############################################################################
# 5. STOKES VECTOR (THIS IS THE UPGRADE YOU ACTUALLY WANT FOR SWEEPS)
###############################################################################

def stokes(E):
    """
    Polarization state in physically meaningful coordinates.
    """

    Ex, Ey = E[0], E[1]

    S0 = simplify(Abs(Ex)**2 + Abs(Ey)**2)
    S1 = simplify(Abs(Ex)**2 - Abs(Ey)**2)
    S2 = simplify(2 * (Ex * Ey.conjugate()).as_real_imag()[0])
    S3 = simplify(-2 * (Ex * Ey.conjugate()).as_real_imag()[1])

    return Matrix([S0, S1, S2, S3])

###############################################################################
# 6. APPLY PIPELINE TO YOUR FIELD
###############################################################################

# Step 1: canonicalize phases
armE_clean = clean_phases(armE)

# Step 2: remove meaningless global phase (VERY IMPORTANT)
armE_phys = remove_global_phase(armE_clean)

# Step 3: phasor view
A, Phi = phasor_decompose(armE_phys)

# Step 4: polarization view
S = stokes(armE_phys)

###############################################################################
# 7. DISPLAY RESULTS
###############################################################################

print("====================================")
print("Amplitude (|E_x|, |E_y|)")
print("====================================")
display(A)

print("====================================")
print("Phase (relative)")
print("====================================")
display(Phi)

print("====================================")
print("Stokes vector [S0, S1, S2, S3]")
print("====================================")
display(S)

Amplitude (|E_x|, |E_y|)


⎡  _____________________     ⎤
⎢╲╱ 2 - 2⋅cos(\phi_{2}) ⋅│aₓ│⎥
⎢────────────────────────────⎥
⎢             2              ⎥
⎢                            ⎥
⎣           │a_y│            ⎦

Phase (relative)


⎡   ⎛                                                             ⎛   ⎛ ⅈ⋅(\ph ↪
⎢   ⎜   ⎛ ⅈ⋅(\phi_{1} + δ)    ⅈ⋅(\phi_{1} + \phi_{2} + δ)⎞  -ⅈ⋅arg⎝aₓ⋅⎝ℯ       ↪
⎢arg⎝aₓ⋅⎝ℯ                 - ℯ                           ⎠⋅ℯ                   ↪
⎢                                                                              ↪
⎢                           ⎛             ⎛   ⎛ ⅈ⋅(\phi_{1} + δ)    ⅈ⋅(\phi_{1 ↪
⎢                           ⎜       -ⅈ⋅arg⎝aₓ⋅⎝ℯ                 - ℯ           ↪
⎣                        arg⎝ⅈ⋅a_y⋅ℯ                                           ↪

↪ i_{1} + δ)    ⅈ⋅(\phi_{1} + \phi_{2} + δ)⎞⎞⎞⎤
↪            - ℯ                           ⎠⎠⎟⎥
↪                                            ⎠⎥
↪                                             ⎥
↪ } + \phi_{2} + δ)⎞⎞⎞                        ⎥
↪                  ⎠⎠⎟                        ⎥
↪                    ⎠                        ⎦

Stokes vector [S0, S1, S2, S3]


⎡        2  ⅈ⋅\phi_{2}     2     2  -ⅈ⋅\phi_{2}           ⎤
⎢      aₓ ⋅ℯ             aₓ    aₓ ⋅ℯ                 2    ⎥
⎢    - ─────────────── + ─── - ──────────────── + a_y     ⎥
⎢             4           2           4                   ⎥
⎢                                                         ⎥
⎢        2  ⅈ⋅\phi_{2}     2     2  -ⅈ⋅\phi_{2}           ⎥
⎢      aₓ ⋅ℯ             aₓ    aₓ ⋅ℯ                 2    ⎥
⎢    - ─────────────── + ─── - ──────────────── - a_y     ⎥
⎢             4           2           4                   ⎥
⎢                                                         ⎥
⎢aₓ⋅a_y⋅(sin(\phi_{1} + δ) - sin(\phi_{1} + \phi_{2} + δ))⎥
⎢                                                         ⎥
⎣aₓ⋅a_y⋅(cos(\phi_{1} + δ) - cos(\phi_{1} + \phi_{2} + δ))⎦

In [99]:
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider
from sympy import lambdify

###############################################################################
# ASSUMPTION:
# You must have a function like:
#
#   armE = arm_function(phi1, phi2, delta)
#
# that returns a SymPy Jones vector.
###############################################################################

# ---- replace this with YOUR system -----------------
def arm_function(phi1, phi2, delta):
    # placeholder example (replace with your real interferometer)
    from sympy import Matrix, exp, I
    return Matrix([
        exp(I*phi1) + exp(I*phi2),
        exp(I*delta)
    ])
# ----------------------------------------------------


###############################################################################
# STOKES (NUMERICAL VERSION)
###############################################################################

def stokes_numeric(E):
    Ex = complex(E[0])
    Ey = complex(E[1])

    S0 = np.abs(Ex)**2 + np.abs(Ey)**2
    S1 = np.abs(Ex)**2 - np.abs(Ey)**2
    S2 = 2 * np.real(Ex * np.conj(Ey))
    S3 = -2 * np.imag(Ex * np.conj(Ey))

    return np.array([S0, S1, S2, S3])


###############################################################################
# NORMALIZE TO POINCARÉ SPHERE
###############################################################################

def normalize_stokes(S):
    S0, S1, S2, S3 = S
    return np.array([S1, S2, S3]) / (S0 + 1e-12)


###############################################################################
# SPHERE PLOT
###############################################################################

def plot_poincare(vec):
    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(111, projection='3d')

    # sphere grid (coarse)
    u = np.linspace(0, 2*np.pi, 30)
    v = np.linspace(0, np.pi, 15)

    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones_like(u), np.cos(v))

    ax.plot_wireframe(x, y, z, color='lightgray', alpha=0.3)

    # point
    ax.scatter(vec[0], vec[1], vec[2], s=120, color='red')

    ax.set_xlim([-1,1])
    ax.set_ylim([-1,1])
    ax.set_zlim([-1,1])

    ax.set_xlabel("S1")
    ax.set_ylabel("S2")
    ax.set_zlabel("S3")

    plt.title("Poincaré Sphere")
    plt.show()


###############################################################################
# INTERACTIVE CELL
###############################################################################

def update(phi1, phi2, delta):

    # evaluate symbolic system
    E = arm_function(phi1, phi2, delta)

    # convert to numeric stokes
    S = stokes_numeric(E)

    # normalize to sphere
    p = normalize_stokes(S)

    # display results
    print("Stokes vector:")
    print(S)

    plot_poincare(p)


interact(
    update,
    phi1=FloatSlider(min=0, max=np.pi, step=0.05, value=0.5, description="phi1"),
    phi2=FloatSlider(min=0, max=np.pi, step=0.05, value=0.5, description="phi2"),
    delta=FloatSlider(min=0, max=np.pi, step=0.05, value=0.5, description="delta")
)

interactive(children=(FloatSlider(value=0.5, description='phi1', max=3.141592653589793, step=0.05), FloatSlide…

<function __main__.update(phi1, phi2, delta)>